In [10]:
import pandas as pd
import calendar
import numpy as np
import datetime as dt
import time
import glob
import os
pd.set_option('display.max_columns',None)
pd.set_option('display.float_format', lambda x: '%.1f' %x)

In [3]:
#Import the Daily Collection Files
Files = glob.glob(r"D:\Strategy and Analytics\Collections\Collections Daily\Consolidated Files\*.csv")

dfs = []
for File in Files:
    df = pd.read_csv(File, dtype='unicode')
    dfs.append(df)
#Combine Files
db = pd.concat(dfs, axis=0, ignore_index=True)
Cols = ['SERVICE_NO', 'REGION', 'DISTRICT', 'TARIFF_CODE','TARIFF', 'SUBTARIFF_CODE', 'SUBTARIFF', 'TARIFF_CLASS', 
 'CONTRACT_STATUS', 'CUSTOMER_NAME','PAYMENT_DATE', 'PROCESSING_DATE','RECEIPT_ID', 'COLLECTED_AMOUNT']

Colsn = ['SERVICE_NO','PAYMENT_DATE']
db1 = db[Colsn].copy()
db1['Count'] = 1
db1['PAYMENT_DATE'] = pd.to_datetime(db1['PAYMENT_DATE'], errors='coerce')
db1['Year'] = db1['PAYMENT_DATE'].dt.year
db1['Month'] = db1['PAYMENT_DATE'].dt.month
db1['Months'] = db1['PAYMENT_DATE'].dt.strftime('%b')
db1['Day'] = db1['PAYMENT_DATE'].dt.day
db1 = db1.sort_values(by='PAYMENT_DATE')
db1['SERVICE_NO'] = db1['SERVICE_NO'].astype(str)
db1 = db1.drop_duplicates()
db1.head(3)

In [7]:
#Define Columns
df_Cols = ['SERVICE_NO','1st', '2nd', '3rd', '4th', '5th', '6th', '7th', '8th', '9th', '10th','11th', '12th', '13th', 
              '14th', '15th', '16th', '17th', '18th', '19th','20th', '21st', '22nd', '23rd', '24th', '25th', '26th', 
              '27th', '28th','29th', '30th', '31st', 'Insts', 'Min_Date', 'Max_Date']
df_Colsn = ['SERVICE_NO', 'Month_Year', 'Insts', 'Min_Date', 'Max_Date', ]
db1['Month_Year'] = db1.apply(lambda row: row['Months'] + "_" + str(row['Year']), axis=1)

Month_Years = []
Months = [month[:3] for month in calendar.month_name[1:]]
for Year in ['2019','2021','2022', '2023']:
    for Month in Months:
        Month_Year = f"{Month}_{Year}"
        Month_Years.append(Month_Year)
Monthsn = Month_Years[:-5]
#Drop_Months = ['Apr_2023','May_2023']
#Monthsn = [Month for Month in Monthsn if Month not in Drop_Months]
Fn = pd.DataFrame(Monthsn,columns=['Month_Year'])
Fn['Sn'] = range(1, len(Fn) + 1)

#Loop Through The Months and rows
dbns = []
for Month in Monthsn:
    #Map Payments
    db2 = db1[db1['Month_Year']==Month]
    db3 = pd.pivot_table(db2, index='SERVICE_NO', columns='PAYMENT_DATE',values='Count')
    db3.fillna(np.nan, inplace=True)
    db3.reset_index(inplace=True)
    #Add Missing Columns
    if len(db3.columns)==29:
        db3[['29th','30th','31st']] = np.nan
    elif len(db3.columns)==30:
        db3[['30th','31st']] = np.nan
    elif len(db3.columns)==31:
        db3['31st'] = np.nan
    else:
        pass
    df = db3.copy() 
    # Get the date columns (excluding the first two columns)
    date_columns = df.columns[1:]
    # Calculate the number of ones in each row
    df["Insts"] = df[date_columns].apply(lambda row: row.count(), axis=1)
    # Find the first and last "1" dates
    df["Min_Date"] = df[date_columns].apply(lambda row: min([col for col, val in zip(date_columns, row) if val == 1]), axis=1)
    df["Max_Date"] = df[date_columns].apply(lambda row: max([col for col, val in zip(date_columns, row) if val == 1]), axis=1)
    # Replace missing values with 0
    df.fillna({"Min_Date": 0, "Max_Date": 0}, inplace=True)
    df[['Min_Date','Max_Date']] = df[['Min_Date','Max_Date']].apply(pd.to_datetime)
    try:
        df['Min_Date'] = df['Min_Date'].dt.day
        df['Max_Date'] = df['Max_Date'].dt.day
    except:
        pass
    df.columns = df_Cols
    df['Month_Year'] = Month
    df.fillna(np.nan,inplace=True)
    df = df[df_Colsn]
    dbns.append(df)

In [13]:
#Minimums
Cb = pd.concat(dbns, ignore_index=True, axis=0)
Cb2 = pd.merge(Cb,Fn, on='Month_Year', how='left')
Pv1 = pd.pivot_table(Cb2, index='SERVICE_NO', columns=['Sn','Month_Year'], values=['Min_Date'])
Pv1 = Pv1.reset_index()
Min_dates = Pv1.fillna(np.nan)
Min_dates.columns = Min_dates.columns.get_level_values('Month_Year')
Min_dates.columns = ['SERVICE_NO'] + list(Min_dates.columns)[1:]
Min_dates['Min_Min_Dates'] = Min_dates.iloc[:,1:].min(axis=1)
Mths = list(Min_dates.columns[1:-1])
Min_dates["Payments"] = Min_dates[Mths].apply(lambda row: row.count(), axis=1)
Min_dates.to_csv(r"D:\Strategy and Analytics\Collections\Collections Daily\Payment Days\Min_dates.csv")
Min_dates.head(5)

,SERVICE_NO,Jan_2019,Feb_2019,Mar_2019,Apr_2019,May_2019,Jun_2019,Jul_2019,Aug_2019,Sep_2019,Oct_2019,Nov_2019,Dec_2019,Jan_2021,Feb_2021,Mar_2021,Apr_2021,May_2021,Jun_2021,Jul_2021,Aug_2021,Sep_2021,Oct_2021,Nov_2021,Dec_2021,Jan_2022,Feb_2022,Mar_2022,Apr_2022,May_2022,Jun_2022,Jul_2022,Aug_2022,Sep_2022,Oct_2022,Nov_2022,Dec_2022,Jan_2023,Feb_2023,Mar_2023,Apr_2023,May_2023,Jun_2023,Jul_2023,Min_Min_Dates,Payments
0,200059530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2
1,200059562,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0,1
2,200059578,14.0,15.0,14.0,12.0,13.0,NaN,12.0,22.0,20.0,14.0,25.0,19.0,7.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,13
3,200059579,30.0,18.0,23.0,29.0,19.0,17.0,23.0,13.0,22.0,21.0,19.0,18.0,31.0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,14
4,200059580,16.0,6.0,14.0,5.0,6.0,7.0,16.0,14.0,11.0,1.0,11.0,11.0,9.0,NaN,NaN,NaN,6.0,14.0,19.0,2.0,2.0,12.0,21.0,4.0,23.0,NaN,NaN,NaN,NaN,28.0,NaN,NaN,16.0,4.0,10.0,14.0,16.0,12.0,7.0,NaN,8.0,NaN,NaN,1.0,31


In [14]:
#Maximums
Pv2 = pd.pivot_table(Cb2, index='SERVICE_NO', columns=['Sn','Month_Year'], values=['Max_Date'])
Pv2 = Pv2.reset_index()
Max_dates = Pv2.fillna(np.nan)
Max_dates.columns = Max_dates.columns.get_level_values('Month_Year')
Max_dates.columns = ['SERVICE_NO'] + list(Max_dates.columns)[1:]
Max_dates['Max_Max_Dates'] = Max_dates.iloc[:,1:].max(axis=1)
Max_dates.to_csv(r"D:\Strategy and Analytics\Collections\Collections Daily\Payment Days\Max_dates.csv")
Max_dates.head(5)

,SERVICE_NO,Jan_2019,Feb_2019,Mar_2019,Apr_2019,May_2019,Jun_2019,Jul_2019,Aug_2019,Sep_2019,Oct_2019,Nov_2019,Dec_2019,Jan_2021,Feb_2021,Mar_2021,Apr_2021,May_2021,Jun_2021,Jul_2021,Aug_2021,Sep_2021,Oct_2021,Nov_2021,Dec_2021,Jan_2022,Feb_2022,Mar_2022,Apr_2022,May_2022,Jun_2022,Jul_2022,Aug_2022,Sep_2022,Oct_2022,Nov_2022,Dec_2022,Jan_2023,Feb_2023,Mar_2023,Apr_2023,May_2023,Jun_2023,Jul_2023,Max_Max_Dates
0,200059530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.0
1,200059562,14.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.0
2,200059578,14.0,15.0,14.0,12.0,13.0,NaN,12.0,22.0,20.0,14.0,25.0,19.0,7.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0
3,200059579,30.0,18.0,23.0,29.0,19.0,17.0,23.0,14.0,22.0,21.0,19.0,18.0,31.0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.0
4,200059580,24.0,6.0,14.0,25.0,30.0,12.0,16.0,14.0,11.0,16.0,17.0,12.0,9.0,NaN,NaN,NaN,18.0,16.0,31.0,31.0,18.0,27.0,21.0,8.0,24.0,NaN,NaN,NaN,NaN,28.0,NaN,NaN,19.0,4.0,19.0,19.0,16.0,28.0,7.0,NaN,8.0,NaN,NaN,31.0


In [15]:
#Installments
Pv3 = pd.pivot_table(Cb2, index='SERVICE_NO', columns=['Sn','Month_Year'], values=['Insts'])
Pv3 = Pv3.reset_index()
Insts = Pv3.fillna(np.nan)
Insts.columns = Insts.columns.get_level_values('Month_Year')
Insts.columns = ['SERVICE_NO'] + list(Insts.columns)[1:]
Insts['Min_Inst'] = Insts.iloc[:,1:].min(axis=1)
Insts['Max_inst'] = Insts.iloc[:,1:].max(axis=1)
Insts.to_csv(r"D:\Strategy and Analytics\Collections\Collections Daily\Payment Days\Installments.csv")
Insts.head(3)

,SERVICE_NO,Jan_2019,Feb_2019,Mar_2019,Apr_2019,May_2019,Jun_2019,Jul_2019,Aug_2019,Sep_2019,Oct_2019,Nov_2019,Dec_2019,Jan_2021,Feb_2021,Mar_2021,Apr_2021,May_2021,Jun_2021,Jul_2021,Aug_2021,Sep_2021,Oct_2021,Nov_2021,Dec_2021,Jan_2022,Feb_2022,Mar_2022,Apr_2022,May_2022,Jun_2022,Jul_2022,Aug_2022,Sep_2022,Oct_2022,Nov_2022,Dec_2022,Jan_2023,Feb_2023,Mar_2023,Apr_2023,May_2023,Jun_2023,Jul_2023,Min_Inst,Max_inst
0,200059530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.0
1,200059562,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0
2,200059578,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0


In [16]:
#Combine Minimuns and Maximums
Combined =  pd.merge(Min_dates[['SERVICE_NO','Payments','Min_Min_Dates']],Max_dates[['SERVICE_NO','Max_Max_Dates']], on='SERVICE_NO', how='left')
Combined =  pd.merge(Combined[['SERVICE_NO','Payments','Min_Min_Dates','Max_Max_Dates']],Insts[['SERVICE_NO','Min_Inst','Max_inst']], on='SERVICE_NO', how='left')
Combined.fillna(np.nan, inplace=True)
Combined[['Min_Min_Dates', 'Max_Max_Dates']] = Combined[['Min_Min_Dates', 'Max_Max_Dates']].astype(int)

#Convert Numbers to Dates
def Convert_to_ordinal(number):
    if number % 10 == 1 and number % 100 != 11:
        return str(number) + 'st'
    elif number % 10 == 2 and number % 100 != 12:
        return str(number) + 'nd'
    elif number % 10 == 3 and number % 100 != 13:
        return str(number) + 'rd'
    else:
        return str(number) + 'th'

Combined['Min_Min_Dates_n'] = Combined['Min_Min_Dates'].apply(Convert_to_ordinal)
Combined['Max_Max_Dates_n'] = Combined['Max_Max_Dates'].apply(Convert_to_ordinal)
Combined.to_csv(r"D:\Strategy and Analytics\Collections\Collections Daily\Payment Days\Combined_Dates_of_Payment.csv", index=False)
Combined.head(3)

,SERVICE_NO,Payments,Min_Min_Dates,Max_Max_Dates,Min_Inst,Max_inst,Min_Min_Dates_n,Max_Max_Dates_n
0,200059530,2,5,22,1.0,2.0,5th,22nd
1,200059562,1,14,14,1.0,1.0,14th,14th
2,200059578,13,7,25,1.0,1.0,7th,25th


# Mid - Month Collections Update Reports

In [25]:
#Pick the Updated Debt File
Month = 'AUG'
try:
    Files = glob.glob(Fr"\\serverfile\Data-Extracts\DISTRICTS\DAILY\COLLECTIONS_REPORT\2023\{Month}\*.csv")
    if 'consolidated' in File:
        Collection = pd.concat([pd.read_csv(File, usecols=Cols, dtype='unicode', encoding='latin-1') for File in Debt_File], ignore_index=True)       
except:
    Files = glob.glob(r"D:\Strategy and Analytics\Collections\Collections Daily\Consolidated Files\*.csv")
    Collection = pd.concat([pd.read_csv(File, usecols=Cols, dtype='unicode', encoding='latin-1') for File in Files], ignore_index=True)

In [26]:
# File path
Month = 'AUG'

#Import Updated Collections File
Filepath = fr"\\serverfile\Data-Extracts\DISTRICTS\DAILY\COLLECTIONS_REPORT\2023\{Month}\*.csv"
#Import Service Points
points = pd.read_csv(r"D:\Strategy and Analytics\Collections\Collections Model\Model_Imports\Collections Reports\Database of Service Points\All_Service_Points_DataBase.csv", dtype='unicode')
#Import payment Dates
dates = pd.read_csv(r"D:\Strategy and Analytics\Collections\Collections Daily\Payment Days\Combined_Payment_Dates.csv", dtype='unicode')

#Billing Cycle
Sales_Cycle = 8

# Obtain the Current date
Today = dt.date.today()
Files = glob.glob(Filepath)

for File in Files:
    if 'consolidated' in File:
        Collections = pd.read_csv(File, dtype='unicode', encoding='latin-1').applymap(lambda x: x.strip() if isinstance(x, str) else x)
        Filename = os.path.splitext(os.path.basename(File))[0]
        Collections.rename(columns={'SERVICE_NO': 'CONTRACT_NO'}, inplace=True)
        Collections['SALE_CYCLE'] = Sales_Cycle
        Cols = ['REGION', 'DISTRICT', 'CONTRACT_NO', 'CUSTOMER_NAME', 'TARIFF', 'TARIFF_CLASS', 'CONTRACT_STATUS', 'SALE_CYCLE', 'BILLED_AMOUNT', 'COLLECTED_AMOUNT']
        Collections = Collections[Cols].dropna(subset=['DISTRICT', 'CONTRACT_NO']).fillna(np.nan).astype({'BILLED_AMOUNT': float, 'COLLECTED_AMOUNT': float})
        #Collections[['BILLED_AMOUNT', 'COLLECTED_AMOUNT']] = Collections[['BILLED_AMOUNT', 'COLLECTED_AMOUNT']].apply(lambda x: x.str.replace(',', ' ').replace('-', '0')).astype(float)

points.rename(columns={'SERVICE_NO': 'CONTRACT_NO'}, inplace=True)
Merged = pd.merge(Collections, points, on='CONTRACT_NO', how='left').fillna(0)
Merged['SP_POINT_NO'] = Merged['SP_POINT_NO'].astype(str)
Merged = Merged[Merged['CONTRACT_NO'].apply(lambda x: len(str(x)) == 9)]
Merged = Merged.drop_duplicates(subset='CONTRACT_NO')
Merged = Merged[Merged['TARIFF_CLASS'].isin(['KVA','TOU','OTHERS'])]
Merged.drop('Month_Year', axis=1, inplace=True)
Dated =  pd.merge(Merged, dates[['SP_POINT_NO','Min_Min_Dates_n','Max_Max_Dates_n']], on='SP_POINT_NO', how='left')
Dated = Dated.drop_duplicates(subset='CONTRACT_NO')
Dated.rename(columns={'Min_Min_Dates_n': 'Min_Pay_Dates', 'Max_Max_Dates_n': 'Max_Pay_Dates'}, inplace=True)
Dated.insert(2, 'SP_POINT_NO', Dated.pop('SP_POINT_NO'))

#Export File
Dated.to_csv(fr"D:\Strategy and Analytics\Collections\Collections Processed Reports\Collections_Report_Update_Cycle_{Sales_Cycle}_{Today}.csv", index=False)
#Collections Summary
Summary = pd.DataFrame({'Details':['Billed','Collected'], 'Amount': [Dated['BILLED_AMOUNT'].sum(axis=0), Dated['COLLECTED_AMOUNT'].sum(axis=0)]})
Summary.head()

,Details,Amount
0,Billed,141041726004.6
1,Collected,99764076654.9


In [27]:
#Print File
Dated.head(5)

,REGION,DISTRICT,SP_POINT_NO,CONTRACT_NO,CUSTOMER_NAME,TARIFF,TARIFF_CLASS,CONTRACT_STATUS,SALE_CYCLE,BILLED_AMOUNT,COLLECTED_AMOUNT,Min_Pay_Dates,Max_Pay_Dates
0,KAMPALA_EAST,CSC_WANDEGEYA,2571726,206116270,PEARL SCIENCES LTD HEALTH,INDUSTRIAL - LV,KVA,ACTIVE,8,822033.4,0.0,2nd,26th
2,KAMPALA_EAST,CSC_WANDEGEYA,2575957,206468932,ENGINEERS INVESTMENTS LTD,COMMERCIAL,TOU,ACTIVE,8,4007598.4,4008000.0,7th,29th
4,KAMPALA_EAST,CSC_KIREKA,2595097,200083267,KYOBE WILSON,DOMESTIC,OTHERS,INACTIVE.,8,642132.4,0.0,1st,31st
5,KAMPALA_EAST,CSC_KIREKA,2595170,200083340,MUSOKE IDIRISA SEGUYA,DOMESTIC,OTHERS,ACTIVE,8,3964.8,0.0,5th,24th
6,KAMPALA_EAST,CSC_WANDEGEYA,2583247,200817629,AMERIK SODHA,COMMERCIAL,TOU,ACTIVE,8,135292.2,120000.0,5th,25th
